#### Env Load

In [47]:
from dotenv import load_dotenv 
load_dotenv(override=True)

True

#### JsonParsing

In [ ]:
import json
from langchain.agents import create_agent

with open("cases.json", "r", encoding="utf-8") as f:
    raw = json.load(f)

# cases.json 구조가 {"cases": [ {...}, {...} ]} 형태이므로
# id를 키로 하는 dict로 변환해서 빠르게 조회할 수 있게 함
cases_db = {case["id"]: case for case in raw["cases"]}

In [145]:
case = cases_db["case_001"]

title = case["title"]
truth = case["truth"]
key_facts = case["key_facts"]
public_scenario = case["public_scenario"]
regression_tests = case["regression_tests"]


print(title)            # "닫힌 화실"
print(truth)            # 진상 전체 텍스트
print(key_facts)        # 핵심 사실 리스트
print(public_scenario)  # 공개된 상황

닫힌 화실
이수아는 한지훈의 미술학원에서 3년간 일한 강사였다. 한지훈은 이수아가 그린 작품을 자신의 이름으로 공모전에 출품해 수상 경력을 쌓아왔다. 이수아는 최근 이 사실을 뒷받침하는 초안 스케치와 날짜가 적힌 메모를 발견했다. 사건 당일, 이수아는 평소처럼 원장에게 차를 대접하며 이 사실을 추궁했고, 한지훈은 오히려 그녀를 해고하겠다고 협박했다. 격분한 이수아는 미리 준비해 온 독을 찻잔에 탔다. 한지훈이 쓰러진 뒤, 이수아는 화실을 나가면서 문을 안에서 잠그는 특수한 방법을 썼다 — 화실 문고리는 스프링식 걸쇠 구조로, 문을 세게 닫으면 자동으로 잠기는 구조였다. 그녀는 이 구조를 누구보다 잘 알고 있었는데, 예전에 원장이 직접 시범을 보여준 적이 있었기 때문이다. 창문은 원래부터 잠겨 있었고 사건과 무관하다. 이수아는 찻잔을 씻지 않고 그대로 두고 나갔는데, 이는 서두르다 실수한 것이지 의도한 것은 아니었다.
['이수아가 범인이다', '동기는 한지훈이 이수아의 그림을 자신의 이름으로 도용해 수상해왔기 때문이다', '독은 찻잔에 탔다', '문은 스프링식 걸쇠 구조라 세게 닫으면 저절로 잠긴다 (밀실 트릭)', '창문은 사건과 무관하다', '찻잔을 치우지 않고 나간 것은 실수였다']
미술학원 원장 한지훈이 화실 안에서 쓰러진 채 발견되었다. 문은 안에서 잠겨 있었고, 창문도 모두 잠겨 있었다. 검시 결과 독극물 중독이었다. 화실에는 그날 사용한 찻잔 하나만 남아 있었다.


#### 판단 노드

In [49]:
from pydantic import BaseModel, Field
from typing import Literal

class AgentResponse(BaseModel):
    answer: Literal["예", "아니오", "상관없음"] = Field(
        description="유저의 질문에 대한 최종 판정"
    )
    evidence: str = Field(description="근거가 된 원문 문장")

In [155]:
detective_prompt = f"""
당신은 예스노 탐정 게임의 진행자입니다.

[플레이어에게 공개된 상황]
{public_scenario}

[비공개 진상 - 플레이어는 모릅니다]
{truth}

[핵심 사실]
{chr(10).join('- ' + f for f in key_facts)}

[규칙]
- 공개된 상황과 비공개 진상을 모두 참고해 판정하세요.
- 위 내용에 명시되지 않았거나 추론할 수 없는 내용은 반드시 "상관없음"으로 답하세요.
- 답변은 "예", "아니오", "상관없음" 중 하나와 근거 문장을 함께 JSON으로 출력하세요.
"""

In [165]:
print(detective_prompt)


당신은 예스노 탐정 게임의 진행자입니다.

[플레이어에게 공개된 상황]
미술학원 원장 한지훈이 화실 안에서 쓰러진 채 발견되었다. 문은 안에서 잠겨 있었고, 창문도 모두 잠겨 있었다. 검시 결과 독극물 중독이었다. 화실에는 그날 사용한 찻잔 하나만 남아 있었다.

[비공개 진상 - 플레이어는 모릅니다]
이수아는 한지훈의 미술학원에서 3년간 일한 강사였다. 한지훈은 이수아가 그린 작품을 자신의 이름으로 공모전에 출품해 수상 경력을 쌓아왔다. 이수아는 최근 이 사실을 뒷받침하는 초안 스케치와 날짜가 적힌 메모를 발견했다. 사건 당일, 이수아는 평소처럼 원장에게 차를 대접하며 이 사실을 추궁했고, 한지훈은 오히려 그녀를 해고하겠다고 협박했다. 격분한 이수아는 미리 준비해 온 독을 찻잔에 탔다. 한지훈이 쓰러진 뒤, 이수아는 화실을 나가면서 문을 안에서 잠그는 특수한 방법을 썼다 — 화실 문고리는 스프링식 걸쇠 구조로, 문을 세게 닫으면 자동으로 잠기는 구조였다. 그녀는 이 구조를 누구보다 잘 알고 있었는데, 예전에 원장이 직접 시범을 보여준 적이 있었기 때문이다. 창문은 원래부터 잠겨 있었고 사건과 무관하다. 이수아는 찻잔을 씻지 않고 그대로 두고 나갔는데, 이는 서두르다 실수한 것이지 의도한 것은 아니었다.

[핵심 사실]
- 이수아가 범인이다
- 동기는 한지훈이 이수아의 그림을 자신의 이름으로 도용해 수상해왔기 때문이다
- 독은 찻잔에 탔다
- 문은 스프링식 걸쇠 구조라 세게 닫으면 저절로 잠긴다 (밀실 트릭)
- 창문은 사건과 무관하다
- 찻잔을 치우지 않고 나간 것은 실수였다

[규칙]
- 공개된 상황과 비공개 진상을 모두 참고해 판정하세요.
- 위 내용에 명시되지 않았거나 추론할 수 없는 내용은 반드시 "상관없음"으로 답하세요.
- 답변은 "예", "아니오", "상관없음" 중 하나와 근거 문장을 함께 JSON으로 출력하세요.



In [156]:
detective_agent = create_agent(
  model='openai:gpt-4.1-mini',
  system_prompt= detective_prompt,
  response_format = AgentResponse,

)

In [157]:
result = detective_agent.invoke({'messages':{'role':'user','content':'김지훈은 엄마가 없어?'}})

In [158]:
question = result['messages'][-2].content
answer = result['structured_response'].answer
evidence = result['structured_response'].evidence

print(question)
print(answer)
print(evidence)


김지훈은 엄마가 없어?
상관없음
김지훈 및 그의 가족 관계에 대한 정보는 주어진 상황과 진상에 포함되어 있지 않습니다.


#### 검증노드

In [161]:
# system: 사건 단위로 고정 (역할, 규칙, 진상, 핵심 사실)
verify_system_prompt = f"""
당신은 추리 게임의 판정 검수자입니다.

[플레이어에게 공개된 상황]
{public_scenario}

[비공개 진상]
{case['truth']}

[핵심 사실]
{chr(10).join('- ' + f for f in case['key_facts'])}

[검토 절차]
1. 먼저 이 질문에 대해 당신이라면 어떻게 답할지 진상만 보고 독립적으로 판단하세요.
2. 당신의 판단과 검토 대상의 답변을 비교하세요.
3. 근거로 제시된 문장이 실제로 진상에 존재하며, 질문과 논리적으로 연결되는지 확인하세요.
4. 진상에 없는 내용을 근거로 들었거나, "상관없음"이어야 할 질문에 단정적으로 답했다면 반드시 오류로 표시하세요.

다음 형식의 JSON으로만 답하세요:
  "is_valid": true|false,
  "corrected_answer": "당신이 판단한 정확한 답 (예|아니오|상관없음)",
  "reason": "검토 대상 답변이 왜 맞거나 틀렸는지 한 문장으로"
"""

# user: 호출마다 바뀌는 데이터 (질문, 1단계 결과)

verify_user_message = f"""
[플레이어의 질문]
{question}

[검토할 판정]
답변: {answer}
근거: {evidence}
"""


In [151]:
print(verify_system_prompt)


당신은 추리 게임의 판정 검수자입니다.

[비공개 진상]
이수아는 한지훈의 미술학원에서 3년간 일한 강사였다. 한지훈은 이수아가 그린 작품을 자신의 이름으로 공모전에 출품해 수상 경력을 쌓아왔다. 이수아는 최근 이 사실을 뒷받침하는 초안 스케치와 날짜가 적힌 메모를 발견했다. 사건 당일, 이수아는 평소처럼 원장에게 차를 대접하며 이 사실을 추궁했고, 한지훈은 오히려 그녀를 해고하겠다고 협박했다. 격분한 이수아는 미리 준비해 온 독을 찻잔에 탔다. 한지훈이 쓰러진 뒤, 이수아는 화실을 나가면서 문을 안에서 잠그는 특수한 방법을 썼다 — 화실 문고리는 스프링식 걸쇠 구조로, 문을 세게 닫으면 자동으로 잠기는 구조였다. 그녀는 이 구조를 누구보다 잘 알고 있었는데, 예전에 원장이 직접 시범을 보여준 적이 있었기 때문이다. 창문은 원래부터 잠겨 있었고 사건과 무관하다. 이수아는 찻잔을 씻지 않고 그대로 두고 나갔는데, 이는 서두르다 실수한 것이지 의도한 것은 아니었다.

[핵심 사실]
- 이수아가 범인이다
- 동기는 한지훈이 이수아의 그림을 자신의 이름으로 도용해 수상해왔기 때문이다
- 독은 찻잔에 탔다
- 문은 스프링식 걸쇠 구조라 세게 닫으면 저절로 잠긴다 (밀실 트릭)
- 창문은 사건과 무관하다
- 찻잔을 치우지 않고 나간 것은 실수였다

[검토 절차]
1. 먼저 이 질문에 대해 당신이라면 어떻게 답할지 진상만 보고 독립적으로 판단하세요.
2. 당신의 판단과 검토 대상의 답변을 비교하세요.
3. 근거로 제시된 문장이 실제로 진상에 존재하며, 질문과 논리적으로 연결되는지 확인하세요.
4. 진상에 없는 내용을 근거로 들었거나, "상관없음"이어야 할 질문에 단정적으로 답했다면 반드시 오류로 표시하세요.

다음 형식의 JSON으로만 답하세요:
  "is_valid": true|false,
  "corrected_answer": "당신이 판단한 정확한 답 (예|아니오|상관없음)",
  "reason": "검토 대상 답변이 왜 맞거나 틀렸는지 한 문장으로"



In [162]:
from langchain.chat_models import init_chat_model

self_consistency_model = init_chat_model(
    "openai:gpt-4.1",
    temperature=0.1,
)

self_consistency_agent = create_agent(
  model=self_consistency_model,
  system_prompt=verify_system_prompt,
)

In [163]:
result2 = self_consistency_agent.invoke({"messages": {"role": "user", "content": verify_user_message}})


In [164]:
print(result2['messages'][-1].content)

{
  "is_valid": true,
  "corrected_answer": "상관없음",
  "reason": "진상에 김지훈이라는 인물이나 그의 가족 관계에 대한 정보가 전혀 없으므로, 질문과 사건의 진상은 무관하다."
}
